# Notebook 13 — The Halting Problem and Universal Computation

**Covers Chapter 5 §5.4–5.6.** The big climaxes of the course:

1. **The Halting Problem** (Theorem 21) — a *concrete* uncomputable predicate. We prove no program can decide whether arbitrary While programs halt on arbitrary inputs.
2. **Universality** — a single program (the *universal program*) can simulate every other While program. Plus the Parameterization Theorem.
3. **The Church-Turing thesis** — the conjecture that all sufficiently powerful computational systems compute the same set of functions.

## What you'll be able to do by the end

- Reproduce the Halting-Problem argument (self-reference + diagonal contradiction).
- Distinguish *deciding* halting from *semi-deciding* halting (one is impossible, one is fine).
- State what the universal partial function ζ is and why it's computable.
- State the Parameterization Theorem and explain its role.
- Exercises 51, 52.

**Heads up:** this notebook is denser-than-usual on theory. The Halting Problem proof requires careful reading.

In [1]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError('could not find while_lang.py')

from while_lang import (
    parse, run, trace, count_steps,
    verify_triple, report_triple,
    StepBudgetExceeded,
)
print('imports OK')

imports OK


## §1. Setting up the Halting Problem

**Background:** Section 6.3 (which we don't have yet) defines a bijection `φ_S : While programs ⇄ ℕ`. So **every program has a unique natural-number index**, and every natural number corresponds to a unique program. We can pass programs around by passing their indices.

**The question:** can we write a program `H` that, given
- the index `m` of some While program `S_m`,
- an input `n`,

outputs `1` if `S_m` terminates on input `n`, and `0` otherwise?

Formally — we'd need `H` to satisfy:

$$\{m = \bar m \in \mathbb{N} \,\land\, n = \bar n \in \mathbb{N}\}\ H\ \{\Downarrow (P \to x = 1) \,\land\, (\neg P \to x = 0)\}$$

where `P` is the predicate "`S_m̄` terminates on input `n̄`".

**Theorem 21 (Halting Problem):** no such `H` exists. The halting predicate is **undecidable**.

Note that we're asking *exactly* the right question. We're not asking "is it hard to write H?" — we're asking "does H exist at all in our computational model?"

## §2. The proof — by contradiction via self-reference

**Suppose** (for contradiction) that `H` exists.

### Step 1 — restrict to the diagonal

Build `H'` from `H` by feeding the same number to both inputs (i.e. asking `H` whether the program with index `m` halts on the input `m` itself):

```
n := m;
H
```

Spec for `H'`:

$$\{m = \bar m \in \mathbb{N}\}\ H'\ \{\Downarrow (P' \to x = 1) \,\land\, (\neg P' \to x = 0)\}$$

where `P'` is "`S_m̄` terminates on input `m̄`" — *self-application* of the program to its own index.

### Step 2 — build a malicious program H''

Construct `H''` to **invert** the answer of `H'` — terminate when `H'` says "won't halt," loop forever when `H'` says "will halt":

```
H';
if x = 1 then
    while tt do skip      // loop forever
else
    x := 1                // terminate, returning 1
```

**Verbal description:** `H''` does the opposite of what `S_m̄` does on input `m̄`.

### Step 3 — Hoare-logic derivation for `H''`

Note we can only assert *partial correctness* for `H''` (it's not guaranteed to terminate by construction):

```
1. {m = m̄ ∈ ℕ} H' {(P' → x = 1) ∧ (¬P' → x = 0)}                  by construction.

2. {(P' → x = 1) ∧ (¬P' → x = 0) ∧ x = 1} skip {(P' → x = 1) ∧ (¬P' → x = 0)}
                                                                    by skip rule.

3. {(P' → x = 1) ∧ (¬P' → x = 0) ∧ x = 1} while tt do skip {¬tt}    by while rule
                                                                    (the post must hold ∧ ¬tt = ff).

4. {(P' → x = 1) ∧ (¬P' → x = 0) ∧ x = 1} while tt do skip {¬P' → x = 1}
                                                                    by consequence (ff implies anything).

5. {(P' → x = 1) ∧ (¬P' → x = 0) ∧ ¬(x = 1)} x := 1 {¬P' → x = 1}
                                                                    by := rule.
                                                                    (Pre forces ¬P' since x = 0.
                                                                    Post: x = 1 and we know ¬P', so ¬P' → x = 1 holds.)

6. {(P' → x = 1) ∧ (¬P' → x = 0)} 
       if x = 1 then (while tt do skip) else x := 1 
       {¬P' → x = 1}                                                from 4, 5 by if rule.

7. {m = m̄ ∈ ℕ} H'' {¬P' → x = 1}                                    from 1, 6 by ; rule.
```

### Step 4 — feed H'' its own index, derive a contradiction

Let `m̄ = φ_S(H'')` (the index of `H''`). Then `P'` becomes:

$$P' = \text{``}H''\text{ terminates on input }\varphi_S(H'')\text{''}$$

Plug into step 7:

$$\{m = \varphi_S(H'')\}\ H''\ \{\neg P' \to x = 1\}$$

**The contradiction.** Two cases:

- **Case A:** `P'` holds — `H''` *does* terminate on its own index. By construction (step 2), this means `H'` returned 1, so `H''` enters the infinite loop. So `H''` *doesn't* terminate. Contradicts `P'`. **CONTRADICTION.**

- **Case B:** `¬P'` — `H''` *doesn't* terminate on its own index. By step 7, *if H'' terminates*, then `x = 1` (since `¬P' → x = 1`). But by construction, when `H'` returns 0 (which corresponds to `¬P'`), we go to the else-branch: `x := 1`. So `H''` *does* terminate. Contradicts `¬P'`. **CONTRADICTION.**

Either case contradicts itself. The original assumption — that `H` exists — must be false. ∎

### Why this works

The trick is **self-reference**: by feeding `H''` its own index, we make `H''` reason about itself. The `if x = 1 then loop else terminate` flips whatever `H` claims, so `H` is forced to predict the *opposite* of what `H''` actually does. No matter what `H` says, it's wrong.

Same shape as Cantor's diagonal argument and Russell's paradox. **"The barber who shaves all and only those who don't shave themselves" — does the barber shave himself?**

## §3. The Halting Problem IS semidecidable (just not decidable)

**The key distinction:** we *can* recognise yes-instances, we just can't recognise no-instances.

**Semi-decision procedure:** given `(m̄, n̄)`, just *run* the program with index `m̄` on input `n̄`:
- If it terminates, output `1`.
- If it doesn't terminate, our procedure also doesn't terminate.

That's exactly what semidecidability requires! Termination of the candidate program ⇔ semi-decider returns yes.

But how do we *run* a program from its index? We need a **universal program** — a single program that can simulate any other program. That's §5.5.

## §4. The partial function computed by a program — Definition 22

Flip the perspective: instead of starting with a function and asking "is there a program?", start with a program and ask "what function does it compute?"

**Definition 22:** the partial function `⟦S⟧ : ℕ ⇀ ℕ` computed by program `S` is given by:

$$\llbracket S \rrbracket(n) = \begin{cases} k & \{n = \bar n \in \mathbb{N}\}\ S\ \{\Downarrow x = k\} \\ \text{undefined} & \text{else}\end{cases}$$

**In words:** if there's a value `k` that S deterministically produces in `x` from input `n` (no matter what other variables hold), then `⟦S⟧(n) = k`. Otherwise, it's undefined.

**Subtle point:** the partial function `⟦S⟧` ignores the values of variables other than `n` at the start. If S's output depends on more than just `n` (e.g. `x := n + y`), then `⟦S⟧` is undefined at every input.

### Examples 18–21

**Example 18:** `x := n` ⟹ `⟦x := n⟧` is the identity.

**Example 19:** the β-program from §4.7.1 ⟹ `⟦β-prog⟧ = β` (total).

**Example 20:** the quadratic-equation program from §4.7.3 — `⟦S⟧(n) = (smallest natural-number solution if any exists, else undefined)`. **Note:** doesn't depend on `n` at all — it's either the constant function returning the solution, or the everywhere-undefined function.

**Example 21:** `x := n + y` ⟹ `⟦S⟧` is undefined everywhere — the output depends on `y`, which the spec doesn't pin down.

### `η_i` notation

For each natural number `i`, we write `η_i` for the partial function computed by the program with index `i`:

$$\eta_i = \llbracket S_i \rrbracket \quad \text{where } S_i \text{ is the program with index } i$$

So `η_0, η_1, η_2, …` is an enumeration of *all* the partial functions computed by While programs.

## §5. The universal partial function — ζ

Define

$$\zeta : \mathbb{N} \times \mathbb{N} \rightharpoonup \mathbb{N}, \quad \zeta(i, j) = \eta_i(j)$$

**Reading:** `ζ(i, j)` = "run the program with index `i` on input `j`, and report what it would output."

If we fix `i`, we get back the partial function computed by program `i`:

$$\zeta(i, -) : \mathbb{N} \rightharpoonup \mathbb{N}, \quad k \mapsto \eta_i(k)$$

**The big claim (§5.5.2):** **`ζ` is itself computable.**

Reading: there's a *single* While program `U` (the **universal program**) such that running `U` on inputs `(i, j)` simulates running program `S_i` on input `j`.

### Why ζ is computable — informal argument

The universal program `U` works like this:

1. **Decode the program.** Take `i`, decode it into the AST of `S_i` using `φ_S⁻¹`. (The encoding `φ_S` is defined in §6.3 — every node is a natural number, with an injective combination of children's indices via Cantor pairing.)
2. **Encode the state.** Represent the state σ as a natural number — also an encoding scheme.
3. **Simulate.** Walk through the AST node-by-node, applying the small-step rules from chapter 2 to update the encoded state.
4. **When `S_i` would terminate** (state's `x` field is some value), decode that value and write it to `U`'s `x`.

**This is hand-wavy.** Building `U` is a substantial coding exercise involving the chapter's encoding functions, but the chapter argues (informally) it's doable. The result is conceptually the same as a Turing-complete interpreter written in its own language.

**Modern parallel:** our `while_lang.py` *is* a universal program. Given a parsed AST and a state, it simulates execution. The chapter's `U` is the same idea, encoded as a single While program.

### Consequence: Halting is semidecidable (formally)

Now we have the machinery. **Semi-decision procedure for Halting:**
```
// input: m̄ (program index), n̄ (input to that program)
x := U(m̄, n̄)   // run universal program on (m̄, n̄)
x := 1          // if we got here, S_{m̄} terminated
```

If `S_{m̄}` halts on `n̄`: U terminates ⟹ we set `x := 1`. ✓  
If `S_{m̄}` doesn't halt: U doesn't halt ⟹ neither do we. **Required for partial-correctness no-claim case.** ✓

So Halting is semidecidable. Combined with Theorem 21 (not decidable), we get an explicit example of a semidecidable-but-not-decidable predicate. **This is what §5.3.3 was setting up.**

## §6. The Parameterization Theorem — Theorem 23

**Statement:** if `f : ℕ × ℕ ⇀ ℕ` is computable, there's a *total* computable function `h : ℕ → ℕ` such that for all `i, j`:

$$f(i, j) = \eta_{h(i)}(j)$$

**Reading:** given any 2-input computable `f`, you can compute (from `i`) the index of a 1-input program that does the same job for that fixed `i`.

### Why it matters

Combined with the universal program: given any 2-input computable function `f` and an `i`, we can
- compute `h(i)` (an index),
- run `U` on `(h(i), j)` to compute `f(i, j)` for any `j`.

**Equivalently:** the universal program is *truly* universal — together with `h`, it can simulate any 2-input computable function. By induction (the s-m-n theorem extension) it works for any number of inputs.

### Sketch of the proof

Given `S` witnessing `f`, construct `S(i)` for each `i`:
```
m := i;       // hardcode i into m
T;            // T computes φ(i, j) into x  (encoding function)
n := x;
S             // computes f(i, j) into x
```

Define `h(i) = φ_S(S(i))` — the index of `S(i)`. The chapter argues this is total computable (because writing the encoding `φ_S` of a program is a finite, mechanical operation).

**The key insight:** generating program *text* from data is itself a computable operation. We can produce indices on demand.

### s-m-n theorem

Generalisation: for any `m, n`, given a computable `f : ℕ^(m+n) ⇀ ℕ` and fixed inputs for the first `m` arguments, we can compute the index of a program for the resulting `n`-arg function. "Fixing parameters is a computable operation."

## §7. The Church-Turing Thesis — Thesis 24

**The thesis (informal):** for any sufficiently powerful computational model, the set of computable functions is the same.

Equivalently: all reasonable formalisations of "effectively computable" agree.

### Models that compute the same set of functions as While

| Model | When proposed | Equivalent to While? |
|:--|:--|:-:|
| **Turing machines** (Turing, 1936) | 1936 | ✓ |
| **λ-calculus** (Church, 1936) | 1936 | ✓ |
| **General recursive functions** (Gödel-Herbrand, Kleene) | 1934 | ✓ |
| **Register machines** (Shepherdson-Sturgis, 1963) | 1963 | ✓ |
| **Cellular automata (Game of Life, etc.)** | 1970 | ✓ |
| **Modern programming languages (Python, C, Lisp, ...)** | various | ✓ (modulo memory) |

All of these compute *exactly* the same set of partial functions ℕ ⇀ ℕ. **The notion of computability is robust.**

### What the thesis is NOT

- It's not a theorem. We can't prove it because "effectively computable" isn't a formal concept — it's an intuition about what humans can do with pencil and paper.
- It's not falsifiable in the usual scientific sense. We could only refute it by exhibiting a model that beats Turing machines, and we'd then have to debate whether that model counts as "effectively computable."
- It doesn't say physical computers can compute *everything* a Turing machine can. Real computers have finite memory.

### What the thesis IS

- A unification claim: all the formalisations agree.
- A practical guide: when proving uncomputability, you can pick whichever model is convenient. The Halting Problem proof works for While *and* Turing machines *and* λ-calculus.
- **Strong evidence:** dozens of independent formalisations have been proposed since 1936, all turning out equivalent.

### Quantum, hypercomputation

**Quantum computing.** Quantum machines can compute things faster (some problems exponentially faster). But the *set* of computable functions is the same as classical — quantum doesn't break Church-Turing.

**Hypercomputation.** Theoretical models with infinite memory or infinite-precision real numbers as inputs can in principle compute things Turing machines can't. **None of these are physically realisable.** They live in the realm of mathematical curiosities.

## §8. Exercises 51, 52

## Exercise 51 — what partial function does each program compute?

Recall: `⟦S⟧(n) = k` iff `{n = n̄ ∈ ℕ} S {⇓ x = k}` holds (as a *valid* triple, regardless of other variables).

### (a)

```
if 100 < n then
    x := n
else
    x := y
```

**Analysis.** For `n > 100`: `x := n` always sets `x = n`, regardless of any other variable. So `⟦S⟧(n) = n` for `n > 100`.

For `n ≤ 100`: `x := y`, but `y` isn't pinned by the precondition. Different starting `y` gives different output. So `⟦S⟧` is undefined for `n ≤ 100`.

$$\llbracket S \rrbracket(n) = \begin{cases} n & n > 100 \\ \text{undefined} & n \le 100 \end{cases}$$

### (b)

```
if 0 ≤ y then
    while 0 < y do y := y − 1
else
    while y < 0 do y := y + 1
x := y
```

**Analysis.** Both branches drive `y` to 0 (the while loops decrement or increment until `y = 0`). Then `x := y = 0`. The output doesn't depend on `n` at all.

Wait — does `n` even appear? No. So the output depends only on `y`'s starting value, both branches make it 0, and `x` ends at 0.

**But:** the partial-function spec requires the value to be the same regardless of other variables. The output is always 0, independent of `y` *and* `n`. So:

$$\llbracket S \rrbracket(n) = 0 \quad \text{for all } n \in \mathbb{N}$$

It's the *constant zero function*.

### (c)

```
while 0 < y do (n := n + n; y := y − 1);
x := n
```

**Analysis.** This loop doubles `n` for each unit of `y`. After `y` iterations, `n` becomes `n · 2^y`.

The output depends on the starting `y`. So the partial function is undefined for all `n`.

$$\llbracket S \rrbracket(n) = \text{undefined for all } n$$

### (d)

```
n := n + 2 × y × y;
while 0 < y × y do (n := n − 2; y := y − 1);
x := n
```

**Analysis.** First `n` becomes `n + 2y²`. Then the loop runs while `y² > 0` (i.e. `y ≠ 0`). Each iteration: `n := n − 2`, `y := y − 1`.

Hmm — `y² > 0` iff `y ≠ 0`. The loop runs `|y|` times, and `y` decrements each iteration. If starting `y > 0`: runs `y` times, `n` decreases by `2y`. Final `n = (n + 2y²) − 2y · 1 − 2(y-1) − ... − 2` — that's not quite right.

Let me re-derive. Start with `n + 2y²` and the loop subtracts 2 from `n` each iteration while decrementing `y`. The loop exits when `y² = 0`, i.e. `y = 0`. If starting `y > 0`: runs `y` iterations, total subtracted: `2y`. Final `n = n_initial + 2y² − 2y`.

So the output depends on starting `y`. Not pinned by `n` alone — undefined.

**Wait** — let me re-check. The precondition only specifies `n = n̄ ∈ ℕ`, not anything about `y`. So `y` could be 0. If starting `y = 0`: first line gives `n := n + 0 = n`, the loop doesn't fire (`y² = 0`), `x := n = n̄`.

If starting `y = 1`: first line `n := n + 2`, loop runs once: `n := n − 2 = n̄`, `y := 0`. Then exit. `x := n̄`. ✓

If starting `y = 2`: `n := n + 8`. Loop: `n := n + 6`, `y := 1`. `n := n + 4`, `y := 0`. Wait — we subtract 2 each iteration, so `n + 8 − 2 = n + 6`. Then `n + 6 − 2 = n + 4`. Then `y² = 0` so exit. `x := n + 4 ≠ n̄`!

Hmm. Let me re-check.

Start `y = 2`, `n = n̄`.  
After `n := n + 2y²` = `n + 8`.  
Loop iteration 1: `y² = 4 > 0`, `n := n − 2 = n + 6`, `y := 1`.  
Loop iteration 2: `y² = 1 > 0`, `n := n − 2 = n + 4`, `y := 0`.  
`y² = 0`, exit.  
`x := n + 4`.

So for `y = 2`: output is `n̄ + 4`. For `y = 0`: output is `n̄`. Different outputs depending on `y`. **Undefined as a function of n alone.**

$$\llbracket S \rrbracket(n) = \text{undefined for all } n$$

(More careful analysis: net change to `n` is `+2y² − 2y = 2y(y − 1)`. For different `y`, this gives different changes, except if we restrict `y ∈ {0, 1}` — but that restriction isn't part of the precondition.)

In [2]:
# Verify our analysis empirically.

# (a) — for n > 100, output should equal n; for n ≤ 100, output depends on y.
prog_a = '''
    if 100 <= n & !(100 = n) then
        x := n
    else (
        x := y
    )
'''
# (We have to encode "100 < n" as "100 ≤ n ∧ ¬(100 = n)".)
for n in [50, 100, 101, 200]:
    for y in [0, 7, 99]:
        out = run(prog_a, {'n': n, 'y': y})
        print(f'  (a) n={n:3}, y={y:3}: x = {out.get("x", 0)}')
    print()

  (a) n= 50, y=  0: x = 0
  (a) n= 50, y=  7: x = 7
  (a) n= 50, y= 99: x = 99

  (a) n=100, y=  0: x = 0
  (a) n=100, y=  7: x = 7
  (a) n=100, y= 99: x = 99

  (a) n=101, y=  0: x = 101
  (a) n=101, y=  7: x = 101
  (a) n=101, y= 99: x = 101

  (a) n=200, y=  0: x = 200
  (a) n=200, y=  7: x = 200
  (a) n=200, y= 99: x = 200



In [3]:
# (b) — output should be 0 regardless of n or y.
prog_b = '''
    if 0 <= y then
        while 1 <= y do (y := y - 1)
    else (
        while y <= -1 do (y := y + 1)
    );
    x := y
'''
# ("y < 0" encoded as "y ≤ -1"; "0 < y" as "1 ≤ y".)
for n in [5, 100]:
    for y in [-3, 0, 7]:
        out = run(prog_b, {'n': n, 'y': y})
        print(f'  (b) n={n:3}, y={y:3}: x = {out.get("x", 0)}   ⟦S⟧ candidate: 0')

  (b) n=  5, y= -3: x = 0   ⟦S⟧ candidate: 0
  (b) n=  5, y=  0: x = 0   ⟦S⟧ candidate: 0
  (b) n=  5, y=  7: x = 0   ⟦S⟧ candidate: 0
  (b) n=100, y= -3: x = 0   ⟦S⟧ candidate: 0
  (b) n=100, y=  0: x = 0   ⟦S⟧ candidate: 0
  (b) n=100, y=  7: x = 0   ⟦S⟧ candidate: 0


## Exercise 52 — choose i so ζ(i, −) equals various functions

> Show that by choosing `i` appropriately, `ζ(i, −)` can equal:
> (a) the identity function on ℕ;
> (b) the constant function returning 2;
> (c) the increment function `n ↦ n + 1`.

### Reading the problem

`ζ(i, −) = η_i = ⟦S_i⟧` — the partial function computed by program with index `i`. So we need to find programs whose partial function is each of these targets, and the desired `i` is the index of that program.

We can't *compute* the actual indices without §6.3's encoding scheme, but we can **exhibit witness programs** and note their existence guarantees an index exists.

### (a) Identity function

Witness program: `x := n`. Its partial function is the identity (Example 18). So `i = φ_S(x := n)`.

From Example 23 in the chapter: `φ_S(x := 0) = 5`. Computing `φ_S(x := n)` requires the same encoding scheme — it's some specific natural number; the chapter doesn't pin it down here.

**Existence claim:** ✓ — such an `i` exists.

### (b) Constant function returning 2

Witness program: `x := 2`. Partial function: returns 2 for every input. **Wait** — does `x := 2` depend on `n`? No, it ignores `n` entirely. By Definition 22, the output for input `n` is `2` (since `{n = n̄ ∈ ℕ} x := 2 {⇓ x = 2}` holds). So `⟦x := 2⟧(n) = 2` for all `n`. ✓

`i = φ_S(x := 2)`. Existence claim: ✓.

### (c) Increment function

Witness program: `x := n + 1`. Partial function: `n ↦ n + 1`. ✓

`i = φ_S(x := n + 1)`. Existence claim: ✓.

### Verifying the partial functions empirically

In [4]:
# Run each witness on a range of inputs.
for label, prog, expected in [
    ('identity',  'x := n',     lambda n: n),
    ('const 2',   'x := 2',     lambda n: 2),
    ('increment', 'x := n + 1', lambda n: n + 1),
]:
    ok = 0
    for n in range(0, 10):
        out = run(prog, {'n': n})
        if out.get('x', 0) == expected(n):
            ok += 1
    print(f'  {label:12}: {ok}/10 inputs match expected partial function')

  identity    : 10/10 inputs match expected partial function
  const 2     : 10/10 inputs match expected partial function
  increment   : 10/10 inputs match expected partial function


## Summary

**Theorem 21 — the Halting Problem is undecidable.** Self-reference + diagonal contradiction. No While program can decide whether arbitrary While programs halt on arbitrary inputs.

**Halting IS semidecidable** — just run the program and report yes when (if) it terminates.

**Universal partial function** `ζ(i, j) = η_i(j)` — "run program i on input j." **Computable** (informally) — there's a single universal program U that simulates every While program by decoding indices and walking the AST step-by-step.

**Parameterization Theorem** (s-1-1 case of s-m-n): given a 2-input computable f, you can compute (from i) the index of a 1-input program for `f(i, −)`. Generating program *text* is a computable operation.

**Church-Turing thesis:** all sufficiently powerful computational models compute the same set of partial functions. Empirically supported by many equivalent formalisations; not a theorem; not falsifiable in the usual scientific sense.

**The big picture across chapter 5:**
- Most functions ℕ → ℕ are uncomputable (cardinality).
- Some predicates are decidable (e.g. primality, evenness).
- Some are *only* semidecidable (e.g. halting).
- Some are neither — even semidecidability fails (e.g. "this program *doesn't* halt on this input" — co-halting).
- The notion of "computable" is robust across formalisations (Church-Turing).

**Next:** N14 quiz tests recognition of decidable / semidecidable / undecidable, the Halting Problem proof structure, and what Definition 22 says about specific programs.